# Mortality: tune models and evaluate on a fixed test set

Compare XGBoost-Cox, elastic-net Cox, and random survival forests using identical baseline and baseline + text cohorts. This notebook creates one stratified 80/20 split, tunes each of the six model variants with 5-fold CV on the training portion only, refits, evaluates the untouched test portion, and records wall-clock time.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'python_scripts' / 'model_training' / 'slurm_array_utils.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the clinical_text_embedding_project repository root.')

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / 'jupyter_notebooks' / 'mortality_model_comparison'
sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(REPO_ROOT / 'python_scripts' / 'model_training'))

from slurm_array_utils import SURV_PATH, build_full_prediction_df, filter_event_rows
from survival_benchmark import (
    DEFAULT_PARAM_GRIDS,
    create_train_test_split,
    prepare_cohort,
    run_train_test_benchmark,
)

pd.set_option('display.max_colwidth', 100)

## Configuration

The default grids deliberately cover a small, interpretable range: 8 XGBoost, 12 elastic-net, and 6 RSF candidates per feature set. Increase `N_JOBS` only to the number of CPUs allocated to this notebook.

In [ ]:
RANDOM_STATE = 1234
N_JOBS = int(os.getenv('SLURM_CPUS_PER_TASK', '1'))
PARAM_GRIDS = DEFAULT_PARAM_GRIDS
OUTPUT_DIR = Path(SURV_PATH) / 'results' / 'death_met_results' / 'mortality_model_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{name: len(__import__('survival_benchmark').expand_grid(grid)) for name, grid in PARAM_GRIDS.items()}

## Build the common mortality cohort

This uses the same `death_met` source, cancer-type merge, baseline columns, and text-column detection as the current full-cohort analysis. Complete cases and constant columns are resolved once before splitting so all six model variants use the same patients.

In [ ]:
full_df, cancer_type_cols, text_cols, events = build_full_prediction_df('death_met')
assert 'death' in events
mortality_df = filter_event_rows(full_df, 'death')
baseline_cols = ['GENDER', 'AGE_AT_TREATMENTSTART'] + cancer_type_cols

cohort, feature_sets = prepare_cohort(
    mortality_df,
    baseline_cols=baseline_cols,
    text_cols=text_cols,
)
cohort_summary = pd.Series({
    'patients': len(cohort),
    'deaths': int(cohort['death'].sum()),
    'censored': int((1 - cohort['death']).sum()),
    'baseline_features': len(feature_sets['baseline']),
    'baseline_text_features': len(feature_sets['baseline_text']),
})
cohort_summary

In [ ]:
split_assignment = create_train_test_split(
    cohort,
    test_size=0.20,
    random_state=RANDOM_STATE,
)
split_check = (
    cohort[['DFCI_MRN', 'death']]
    .merge(split_assignment, on='DFCI_MRN', validate='one_to_one')
    .groupby('split')['death']
    .agg(['size', 'sum', 'mean'])
)
split_check

## Tune on 80%, refit, and evaluate on 20%

Selection uses mean cumulative/dynamic AUC across the five training folds. The saved timing separates hyperparameter search, final refit, and prediction/evaluation, while `total_seconds` is the requested end-to-end wall time.

In [ ]:
summary, best_hyperparameters = run_train_test_benchmark(
    cohort,
    feature_sets,
    split_assignment,
    OUTPUT_DIR,
    param_grids=PARAM_GRIDS,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
summary.sort_values(['model', 'feature_set'])

In [ ]:
display_cols = [
    'model', 'feature_set', 'mean_auc_t', 'c_index',
    'integrated_brier_score', 'total_seconds', 'best_params_json'
]
display(summary[display_cols].sort_values(['model', 'feature_set']))

plot_df = summary.pivot(index='model', columns='feature_set', values=['mean_auc_t', 'c_index', 'integrated_brier_score'])
plot_df.plot.bar(subplots=True, layout=(1, 3), figsize=(15, 4), legend=True, title=['Mean AUC(t)', 'C-index', 'Integrated Brier score'])
plt.tight_layout();

## Reproducibility record

In [ ]:
versions = {
    package: metadata.version(package)
    for package in ['numpy', 'pandas', 'scikit-learn', 'scikit-survival', 'xgboost']
}
versions.update({'random_state': RANDOM_STATE, 'n_jobs': N_JOBS})
with (OUTPUT_DIR / 'software_versions.json').open('w') as handle:
    json.dump(versions, handle, indent=2, sort_keys=True)
versions